# Eaton Fire Dataset Exploration

End-to-end exploration of the labeled street view dataset.
Covers:
1. Parse metadata from filenames
2. Class distribution
3. Spatial distribution of points
4. Sample image viewer
5. Inter-class distance analysis (are damage classes spatially clustered?)

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

# Adjust this path if running from outside the repo root
REPO_ROOT = Path("..")
sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT.resolve())

## 1. Parse Metadata

In [ ]:
DAMAGE_CLASSES = {
    "No_Damage":      0,
    "Affected_1-9_":  1,
    "Destroyed_50_": 2,
}
LABEL_NAMES = {0: "No Damage", 1: "Affected (1-9%)", 2: "Destroyed (≥50%)"}
FILENAME_RE = re.compile(
    r"(?P<lat>-?[\d.]+)_(?P<lon>-?[\d.]+)_OID(?P<oid>\d+)_A(?P<aid>\d+)\.jpg",
    re.IGNORECASE,
)

records = []
for cls_name, label in DAMAGE_CLASSES.items():
    folder = REPO_ROOT / "Eaton_Fire" / cls_name / "attachments"
    for fname in sorted(os.listdir(folder)):
        m = FILENAME_RE.match(fname)
        if m:
            records.append({
                "filename": fname,
                "filepath": str(folder / fname),
                "latitude":  float(m.group("lat")),
                "longitude": float(m.group("lon")),
                "object_id": int(m.group("oid")),
                "damage_class": cls_name,
                "label": label,
                "label_name": LABEL_NAMES[label],
            })

df = pd.DataFrame(records)
print(f"Total images: {len(df)}")
df.head()

## 2. Class Distribution

In [ ]:
CLASS_COLORS = {"No_Damage": "#2ecc71", "Affected_1-9_": "#f39c12", "Destroyed_50_": "#e74c3c"}

counts = df["damage_class"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = [CLASS_COLORS[c] for c in counts.index]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white")
axes[0].set_title("Image Count per Class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=15)

axes[1].pie(
    counts.values, labels=counts.index, colors=colors,
    autopct="%1.1f%%", startangle=140, pctdistance=0.75
)
axes[1].set_title("Class Distribution")

plt.tight_layout()
plt.show()
print(counts)

## 3. Spatial Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for cls_name, color in CLASS_COLORS.items():
    subset = df[df["damage_class"] == cls_name]
    ax.scatter(
        subset["longitude"], subset["latitude"],
        c=color, label=LABEL_NAMES[DAMAGE_CLASSES[cls_name]],
        alpha=0.7, s=18, edgecolors="none"
    )

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Spatial Distribution of Labeled Street View Points\n(Altadena, CA — Eaton Fire 2025)")
ax.legend(loc="upper right")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Bounding box:")
print(f"  Lat: [{df['latitude'].min():.5f}, {df['latitude'].max():.5f}]")
print(f"  Lon: [{df['longitude'].min():.5f}, {df['longitude'].max():.5f}]")

## 4. Sample Image Viewer

In [ ]:
N_PER_CLASS = 4
SEED = 42

fig, axes = plt.subplots(3, N_PER_CLASS, figsize=(N_PER_CLASS * 3.5, 9))
fig.suptitle("Sample Street View Images by Damage Class", fontsize=13, y=1.01)

for row_i, (cls_name, color) in enumerate(CLASS_COLORS.items()):
    subset = df[df["damage_class"] == cls_name].sample(
        min(N_PER_CLASS, len(df[df["damage_class"] == cls_name])),
        random_state=SEED
    )
    for col_i in range(N_PER_CLASS):
        ax = axes[row_i, col_i]
        if col_i < len(subset):
            row = subset.iloc[col_i]
            img = Image.open(row["filepath"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"({row['latitude']:.4f},\n{row['longitude']:.4f})", fontsize=7)
        else:
            ax.set_facecolor("#f5f5f5")
        ax.axis("off")
        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_edgecolor(color)
            sp.set_linewidth(2.5)
    axes[row_i, 0].set_ylabel(LABEL_NAMES[DAMAGE_CLASSES[cls_name]], color=color, fontsize=10)

plt.tight_layout()
plt.show()

## 5. Spatial Proximity Analysis

Are destroyed buildings concentrated in specific sub-areas, or spatially mixed with intact ones?
A KDE plot per class answers this.

In [ ]:
import scipy.stats as stats

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

# Extent from full dataset
lon_min, lon_max = df["longitude"].min() - 0.005, df["longitude"].max() + 0.005
lat_min, lat_max = df["latitude"].min() - 0.005, df["latitude"].max() + 0.005
xx, yy = np.mgrid[lon_min:lon_max:200j, lat_min:lat_max:200j]
positions = np.vstack([xx.ravel(), yy.ravel()])

for ax, (cls_name, color) in zip(axes, CLASS_COLORS.items()):
    subset = df[df["damage_class"] == cls_name]
    if len(subset) >= 2:
        values = np.vstack([subset["longitude"].values, subset["latitude"].values])
        kernel = stats.gaussian_kde(values)
        z = kernel(positions).reshape(xx.shape)
        ax.contourf(xx, yy, z.T, levels=12, cmap="Reds" if cls_name == "Destroyed_50_" else "Greens")
    ax.scatter(subset["longitude"], subset["latitude"], s=6, c="white", alpha=0.5)
    ax.set_title(f"{LABEL_NAMES[DAMAGE_CLASSES[cls_name]]}\n(n={len(subset)})", color=color)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")

axes[0].set_ylabel("Latitude")
plt.suptitle("Spatial Density (KDE) by Damage Class", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Next Steps

- Run `scripts/data_prep/parse_metadata.py` to save the CSV
- Run `scripts/data_prep/train_val_test_split.py` to create train/val/test splits
- Run `scripts/features/extract_clip_embeddings.py` to compute CLIP embeddings
- Run `scripts/features/sample_raster_values.py` to attach burn severity (dNBR) values
- Open `notebooks/02_fusion_model.ipynb` (TODO) for the multi-modal model